In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
import logging
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import re
import json


c:\Users\NOVA\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
LOG_DIR = BASE_DIR / "log"
OUT_DIR = DATA_DIR
INPUT_FILE = DATA_DIR / "final_delivery.jsonl"

In [3]:
df = pd.read_json(INPUT_FILE, lines=True)
df.head()

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,collectionsCategory,fAudio,fVideo,imgUrl,isHighQuality,museumName,threeUrl,raw_excavation_info,raw_size_in_desc,other_info
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径19.2,"[{'unit': 'cm', 'value': 19.2, 'deviation': No...",夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,"[{'label': 'material', 'value': '夹砂红陶'}, {'lab...","{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/thumbs/ef3b...,1,济宁市兖州区博物馆,None,None,None,None
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径10.5厘米,"[{'unit': 'cm', 'value': 10.5, 'deviation': No...",None,"[{'label': 'material', 'value': '褐陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,胶州市博物馆,None,None,None,None
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",高*口径*底径：15.3*7.5*6.2,"[{'unit': 'cm', 'value': 15.3, 'deviation': No...",None,"[{'label': 'material', 'value': '红陶'}]","{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,桓台县博物馆,None,None,None,None
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,"{'year': None, 'month': None, 'day': None}",口径13，底径7，高5,"[{'unit': 'cm', 'value': 13.0, 'deviation': No...",None,"[{'label': 'material', 'value': '夹砂黑陶'}, {'lab...","{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/thumbs/d4f9...,1,青岛市黄岛区博物馆,None,None,None,None
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",通高*腹围*口径*流长：39.5*37*9.1*9.9,"[{'unit': 'cm', 'value': 39.5, 'deviation': No...",None,[],"{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandongnew...,1,莒南县博物馆,None,None,None,None


In [4]:
# 为每行的sourceCitation列中的locator字段赋值
# 该值即文物详情页的url，形式为`f"http://www.wwsdw.net/#/collect/detail?id={row["id"]}"`
df_processed = df
df_processed['sourceCitation'] = df_processed.apply(
    lambda row: {**row['sourceCitation'], 'locator': f"http://www.wwsdw.net/#/collect/detail?id={row['id']}", 'locatorType': "URL"}, 
    axis=1
)
df_processed.head()


,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,collectionsCategory,fAudio,fVideo,imgUrl,isHighQuality,museumName,threeUrl,raw_excavation_info,raw_size_in_desc,other_info
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径19.2,"[{'unit': 'cm', 'value': 19.2, 'deviation': No...",夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,"[{'label': 'material', 'value': '夹砂红陶'}, {'lab...","{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/thumbs/ef3b...,1,济宁市兖州区博物馆,None,None,None,None
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径10.5厘米,"[{'unit': 'cm', 'value': 10.5, 'deviation': No...",None,"[{'label': 'material', 'value': '褐陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,胶州市博物馆,None,None,None,None
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",高*口径*底径：15.3*7.5*6.2,"[{'unit': 'cm', 'value': 15.3, 'deviation': No...",None,"[{'label': 'material', 'value': '红陶'}]","{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,桓台县博物馆,None,None,None,None
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,"{'year': None, 'month': None, 'day': None}",口径13，底径7，高5,"[{'unit': 'cm', 'value': 13.0, 'deviation': No...",None,"[{'label': 'material', 'value': '夹砂黑陶'}, {'lab...","{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/thumbs/d4f9...,1,青岛市黄岛区博物馆,None,None,None,None
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",通高*腹围*口径*流长：39.5*37*9.1*9.9,"[{'unit': 'cm', 'value': 39.5, 'deviation': No...",None,[],"{'country': None, 'province': None, 'city': No...",...,2,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandongnew...,1,莒南县博物馆,None,None,None,None


In [5]:
# 现要对每件文物按照特定api格式发起请求，获取它的高分辨率图片url，并将这些url添加到onlineImageUrls列中
# 请求格式：http://www.wwsdw.net/admin/collection/getCollectionById.do?_t={时间戳（秒）}&userId=&token=&id={文物id}
# 返回内容为一个json，要关注其中的data字段下的collectionInfo字段下的pics字段，该字段是一个列表，列表中的每个元素是一个对象，里面包含着丰富的信息，先把这个列表全部爬取下来，后续再从中提取高分辨率图片的url

In [6]:


# # ==========================================
# # 1. 路径与全局配置
# # ==========================================

# 【重要提醒】为了实现断点续传（重复运行），输出的数据文件不应包含时间戳，
# 否则每次运行产生新文件，程序无法读取历史进度。时间戳应仅用于日志文件。
PICS_DATA_OUTPUT = OUT_DIR / "pics_data_output.jsonl"

# 日志文件包含时间戳，方便追溯单次运行的详情
current_time = time.strftime("%Y%m%d_%H%M%S")
LOG_FILE = LOG_DIR / f"crawler_pics_{current_time}.log"

# # ==========================================
# # 2. 日志系统配置
# # ==========================================
# logging.basicConfig(
#     level=logging.INFO,
#     format='%(asctime)s - %(levelname)s - %(message)s',
#     handlers=[
#         logging.FileHandler(LOG_FILE, encoding='utf-8')
#         # 如果需要控制台也输出日志，取消下一行的注释。
#         # 但在 Notebook 中配合 tqdm 可能会导致屏幕混乱，建议只写文件。
#         # , logging.StreamHandler()
#     ]
# )
# logger = logging.getLogger(__name__)

# ==========================================
# 3. 核心请求逻辑
# ==========================================
def fetch_pic_info(artifact_id):
    """
    针对单个文物 id 发起请求，提取 pics 字段。
    返回: (文物id, 是否成功(bool), 提取的数据列表/None, 错误信息/None)
    """
    url = "http://www.wwsdw.net/admin/collection/getCollectionById.do"
    params = {
        "_t": int(time.time()),
        "userId": "",
        "token": "",
        "id": artifact_id
    }
    headers = {
        # 伪装常用浏览器，降低被拦截概率
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    try:
        # 必须设置 timeout，防止个别请求死锁导致线程池耗尽
        response = requests.get(url, params=params, headers=headers, timeout=60)
        response.raise_for_status() 

        data = response.json()
        
        # 安全地逐层提取，防止因响应结构改变导致 KeyError
        pics = data.get("data", {}).get("collectionInfo", {}).get("pics", [])
        
        return str(artifact_id), True, pics, None
    except Exception as e:
        return str(artifact_id), False, None, str(e)

# ==========================================
# 4. 主流程：状态恢复与并行爬取
# ==========================================
def run_crawler(df):
    # 步骤 A：读取已有数据，建立已完成集合，避免重复劳动
    fetched_ids = set()
    if PICS_DATA_OUTPUT.exists():
        with open(PICS_DATA_OUTPUT, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        record = json.loads(line)
                        fetched_ids.add(str(record["id"]))
                    except json.JSONDecodeError:
                        continue

    logger.info(f"状态恢复：发现 {len(fetched_ids)} 条历史记录。")

    # 步骤 B：对比全量数据，筛选出缺失的 ID
    # 确保 ID 格式统一为字符串，避免类型匹配错误
    all_ids = df["id"].dropna().astype(str).unique().tolist()
    pending_ids = [aid for aid in all_ids if aid not in fetched_ids]

    if not pending_ids:
        print("🎉 所有数据均已爬取完毕，无需执行！")
        return

    print(f"总计 {len(all_ids)} 条，已完成 {len(fetched_ids)} 条，本轮待处理 {len(pending_ids)} 条。")
    logger.info(f"开始执行爬取任务，待处理数量: {len(pending_ids)}")

    # 步骤 C：并行爬取与实时落盘
    max_workers = 15 # 4000条数据，15个线程并发是个相对温和且高效的数字
    success_count = 0
    fail_count = 0

    # 使用追加模式 ("a") 打开文件，主线程负责统筹写入，避免多线程写入冲突
    with open(PICS_DATA_OUTPUT, "a", encoding="utf-8") as out_file:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # 将所有未完成的任务提交到线程池
            future_to_id = {executor.submit(fetch_pic_info, aid): aid for aid in pending_ids}

            # as_completed 会在任务完成后立即返回（无序），配合 tqdm 显示进度
            for future in tqdm(as_completed(future_to_id), total=len(pending_ids), desc="爬取进度"):
                artifact_id = future_to_id[future]
                try:
                    aid, success, pics, error_msg = future.result()

                    if success:
                        record = {"id": aid, "pics": pics}
                        # 实时写入单行 JSONL 并强制刷新缓冲区
                        out_file.write(json.dumps(record, ensure_ascii=False) + "\n")
                        out_file.flush()
                        
                        success_count += 1
                        logger.info(f"成功 | ID: {aid} | 提取图片记录数: {len(pics)}")
                    else:
                        fail_count += 1
                        logger.error(f"失败 | ID: {aid} | 错误: {error_msg}")

                except Exception as e:
                    fail_count += 1
                    logger.error(f"线程崩溃 | ID: {artifact_id} | 错误: {str(e)}")

    # 步骤 D：结果汇报
    print(f"\n本轮任务结束。成功: {success_count} 条，失败: {fail_count} 条。")
    print(f"日志已保存至: {LOG_FILE}")
    
    if fail_count > 0:
        print("\n⚠️ 存在失败请求。请稍作等待后，直接重新运行当前单元格即可进行补漏，已成功的数据会被自动跳过。")


In [7]:

# ==========================================
# 5. 执行命令
# ==========================================
# 传入内存中已有的 df_processed 即可启动
# run_crawler(df_processed)

In [8]:
# 读取爬到的数据，拼接到原本的df_processed中，检查前几行
df_pics = pd.read_json(PICS_DATA_OUTPUT, lines=True)
df_merged = df_processed.merge(df_pics, on="id", how="outer")
df_merged.head(10)

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,fAudio,fVideo,imgUrl,isHighQuality,museumName,threeUrl,raw_excavation_info,raw_size_in_desc,other_info,pics
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径19.2,"[{'unit': 'cm', 'value': 19.2, 'deviation': No...",夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,"[{'label': 'material', 'value': '夹砂红陶'}, {'lab...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/thumbs/ef3b...,1,济宁市兖州区博物馆,None,None,None,None,"[{'id': 'f415bfc279cc411a96d354ec220aecf0', 'o..."
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径10.5厘米,"[{'unit': 'cm', 'value': 10.5, 'deviation': No...",None,"[{'label': 'material', 'value': '褐陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,胶州市博物馆,None,None,None,None,[{'id': 'e23f521d-556e-4ba4-8566-af692a6dcb37'...
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",高*口径*底径：15.3*7.5*6.2,"[{'unit': 'cm', 'value': 15.3, 'deviation': No...",None,"[{'label': 'material', 'value': '红陶'}]","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,桓台县博物馆,None,None,None,None,[{'id': '44144ea9-053d-4c93-947a-950b542a43a4'...
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,"{'year': None, 'month': None, 'day': None}",口径13，底径7，高5,"[{'unit': 'cm', 'value': 13.0, 'deviation': No...",None,"[{'label': 'material', 'value': '夹砂黑陶'}, {'lab...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/thumbs/d4f9...,1,青岛市黄岛区博物馆,None,None,None,None,[{'id': '7c1df671-ed26-4c3e-bf92-682fe10c43f6'...
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",通高*腹围*口径*流长：39.5*37*9.1*9.9,"[{'unit': 'cm', 'value': 39.5, 'deviation': No...",None,[],"{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandongnew...,1,莒南县博物馆,None,None,None,None,"[{'id': '1025b8b89e134b449abfac567fc59787', 'o..."
5,0080D7CDCAED42E99F3186AA3816DD20,新石器时代大汶口文化红褐陶盆,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径22.9底径11.5,"[{'unit': 'cm', 'value': 22.9, 'deviation': No...",None,"[{'label': 'material', 'value': '红褐陶'}, {'labe...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,枣庄市博物馆,None,None,None,None,NaN
6,00C5FA0BD4F943D6BA9D20C6A9D27C5E,新石器时代龙山文化灰陶罐,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",口径*底径：14.5*10.7,"[{'unit': 'cm', 'value': 14.5, 'deviation': No...",None,"[{'label': 'material', 'value': '灰陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,桓台县博物馆,None,None,None,None,"[{'id': 'd0c9b62a7f2b44fe91ab4992119b6769', 'o..."
7,00DB2EE22C974FA0AEDE4EEE468A619D,新石器时代大汶口文化夹砂红褐陶瓶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径7.3腹径6.3足径5.24,"[{'unit': 'cm', 'value': 7.3, 'deviation': Non...",None,"[{'label': 'material', 'value': '夹砂红褐陶'}, {'la...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,枣庄市博物馆,None,None,None,None,"[{'id': '9dc9120918b5471f95be1b840b22d851', 'o..."
8,00E0F7EDB6BD4FE18CD60F7DF1074CD9,新石器时代大汶口文化镂孔白陶豆,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径19.30底径12.90,"[{'unit': 'cm', 'value': 19.3, 'deviation': No...",None,"[{'label': 'material', 'value': '白陶'}, {'label...","{'country': None, 'province

In [9]:
# pics是一个字典列表，每个字典都有以下的一些key
# 'id'图片id, 'objectId'文物id, 'typeId', 'gsNo', 'name', 'url', 'picWidth', 'picHeight', 'type', 'isMain', 'status', 'sequence', 'createtime',
# 'thumb1', 'thumb1Width', 'thumb1Height', 'thumb2', 'thumb2Width', 'thumb2Height', 'thumb3', 'thumb3Width', 'thumb3Height', 'thumb4', 'thumb4Width', 'thumb4Height', 'thumb5', 'thumb5Width', 'thumb5Height', 'thumb6', 'thumb6Width', 'thumb6Height'


In [10]:
# 发现有26行的pics是空列表，返回去找它们是哪些文物所对应的行，这些文物的详情页能否正常访问，是否存在图片。
# 另外仅有id为0080D7CDCAED42E99F3186AA3816DD20的文物的pics信息直接是空的，这是唯一有此问题的文物。将其pic先置为空列表。
# 注意：如果用布尔索引+列名的方式选取出需要修改的单元格，那么会遇到一个问题：选取得到的是子表格，如果将空列表[]赋值给它，python或pandas会从广播机制的角度理解，并认为这一操作非常奇怪，无法正常进行。（如果是把确定的值，例如数字1，赋给子表格，可能是可以进行的，但是把空列表[]赋给子表格就是令人困惑的）
# 一个比较好的解决方案是用apply重建这一列，将筛选逻辑写在apply里，如下。
df_merged["pics"] = df_merged["pics"].apply(lambda x : x if isinstance(x, list) else [])
df_merged[df_merged["pics"].apply(lambda x : isinstance(x, list) and len(x) == 0)]
# 观察发现：pics信息为空列表的26行，就是之前那些缺乏imgUrl（主图）的行，点进其中一个的详情页，发现确实没有图，在列表页搜索它，也没有图。
# 而无法访问到pics的那一行，实际上之前爬到过它的数据，但现在再次去看，发现它的详情页干脆访问不了，列表页也搜不到，似乎不再在站上展示；然而奇怪的是，曾经爬到过它的图片，其url竟然还能访问。但现在确实访问不了。由于只有一行如此，可以直接不去管它，将其无视。

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,fAudio,fVideo,imgUrl,isHighQuality,museumName,threeUrl,raw_excavation_info,raw_size_in_desc,other_info,pics
5,0080D7CDCAED42E99F3186AA3816DD20,新石器时代大汶口文化红褐陶盆,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径22.9底径11.5,"[{'unit': 'cm', 'value': 22.9, 'deviation': No...",None,"[{'label': 'material', 'value': '红褐陶'}, {'labe...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,http://www.wwsdw.net/sdimg/picture/shandong/37...,1,枣庄市博物馆,None,None,None,None,[]
116,07754cd41d944954aecde1e89aafe727,新石器时代大汶口文化红陶鼎,None,None,"{'year': None, 'month': None, 'day': None}",口径：6.4,"[{'unit': 'cm', 'value': 6.4, 'deviation': Non...",None,"[{'label': 'material', 'value': '红陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,None,1,东营市历史博物馆,None,None,None,None,[]
253,0a7301cc87914346889af62cba430588,新石器时代大汶口文化红陶罐形鼎,None,None,"{'year': None, 'month': None, 'day': None}",口径：13.4,"[{'unit': 'cm', 'value': 13.4, 'deviation': No...",None,"[{'label': 'material', 'value': '红陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,None,1,东营市历史博物馆,None,None,None,新石器时代大汶口文化,[]
254,0acd48bd7ec74a01b6516bb5740e3893,新石器时代大汶口文化红陶鼎,None,None,"{'year': None, 'month': None, 'day': None}",口径：12.5,"[{'unit': 'cm', 'value': 12.5, 'deviation': No...",器物呈鼎形，通体红色，侈口束颈，折腹，三足，足呈铲形，其中一足断为两截。,"[{'label': 'material', 'value': '红陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,None,1,东营市历史博物馆,None,None,None,None,[]
527,1b863cb8e914429f9042f2de942e9297,新石器时代黑陶杯,None,None,"{'year': None, 'month': None, 'day': None}",口径*底径：7.4*6.3,"[{'unit': 'cm', 'value': 7.4, 'deviation': Non...",None,"[{'label': 'color', 'value': '黑陶'}]","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,None,1,东营市历史博物馆,None,None,None,None,[]
648,264a1c181c2e45039db516c4f474df8b,新石器时代红陶罐形鼎,None,None,"{'year': None, 'month': None, 'day': None}",口径：10.2,"[{'unit': 'cm', 'value': 10.2, 'deviation': No...",None,"[{'label': 'material', 'value': '红陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,None,1,东营市历史博物馆,None,None,None,None,[]
702,29e37036624a48a687a3103e5b1f701a,新石器时代大汶口文化红陶单把壶,None,None,"{'year': None, 'month': None, 'day': None}",口径*底径：11.7*5,"[{'unit': 'cm', 'value': 11.7, 'deviation': No...",None,"[{'label': 'material', 'value': '红陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,None,1,东营市历史博物馆,None,None,None,None,[]
1251,47248623877440fc8b7df810577ceace,新石器时代彩陶鼎,None,None,"{'year': None, 'month': None, 'day': None}",口径：11.7,"[{'unit': 'cm', 'value': 11.7, 'deviation': No...",None,[],"{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,None,1,东营市历史博物馆,None,None,None,None,[]
1418,51c9806ebe6a4f7f98cfea3fcb64ccbb,新石器时代大汶口文化彩绘陶盂,None,None,"{'year': None, 'month': None, 'day': None}",口径*底径：16*11.5,"[{'unit': 'cm', 'value': 16.0, 'deviation': No...",None,[],"{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,None,1,东营市历史博物馆,None,None,None,None,[]
1448,54253223dfa4494cbc3d61d3423ad38a,新石器时代龙山文化红陶鬶,None,None,"{'year': None, 'month': None, 'day': None}",口径：9,"[{'unit': 'cm', 'value': 9.0, 'deviation': Non...",None,"[{'label': 'material', 'value': '红陶'}, {'label...","{'country': None, 'province': None, 'city': No...",...,http://www.wwsdw.net/sdimg/,NaN,None,1,东营市历史博物馆,None,None,None,None,[]


In [11]:
# 总而言之，可以不去管那26+1=27行，只关注4119-27=4092行的pics信息。
# 观察图片信息的性质。将所有图片展开为单行，观察每行的性质。
df_pics_exploded = df_merged[["id","pics"]].explode('pics', ignore_index=True)


In [12]:
df_pics_normalized = pd.json_normalize(df_pics_exploded['pics'])
df_pics_normalized.rename(columns={"id": "picId"}, inplace=True)
df_pics_normalized = pd.concat([df_pics_exploded[["id"]], df_pics_normalized], axis=1)
df_pics_normalized

,id,picId,objectId,typeId,gsNo,name,url,picWidth,picHeight,type,...,thumb3Height,thumb4,thumb4Width,thumb4Height,thumb5,thumb5Width,thumb5Height,thumb6,thumb6Width,thumb6Height
0,003895c08aa84c2a98d3d1696db7ffcf,f415bfc279cc411a96d354ec220aecf0,003895c08aa84c2a98d3d1696db7ffcf,3,,987.3.14-A-01.jpg,https://wx.wwsdw.net/sdimg/picture/thumbs/d40b...,3780.0,2504.0,0.0,...,127.0,,0.0,0.0,,0.0,0.0,back/picture/d40b5bdb48a54875b783d9beefa289d0.JPG,0.0,0.0
1,003895c08aa84c2a98d3d1696db7ffcf,85d94a471441437f84fe0f6fa1a88675,003895c08aa84c2a98d3d1696db7ffcf,3,,987.3.14-E-02.jpg,https://wx.wwsdw.net/sdimg/picture/thumbs/19df...,3872.0,2592.0,0.0,...,0.0,,0.0,0.0,,0.0,0.0,back/picture/19dfa463937a4960ba3027230ab7301c.JPG,0.0,0.0
2,003895c08aa84c2a98d3d1696db7ffcf,765f81b3810c475a8888d40062ad9ae3,003895c08aa84c2a98d3d1696db7ffcf,3,,987.3.14-E-01.jpg,https://wx.wwsdw.net/sdimg/picture/thumbs/2012...,3872.0,2592.0,0.0,...,0.0,,0.0,0.0,,0.0,0.0,back/picture/20121d4a65444b4ca6a00f69ca531613.JPG,0.0,0.0
3,003895c08aa84c2a98d3d1696db7ffcf,9a5ff57108004ae8afe477d03644e1e2,003895c08aa84c2a98d3d1696db7ffcf,3,,987.3.14-F-01.jpg,https://wx.wwsdw.net/sdimg/picture/thumbs/951e...,3872.0,2592.0,0.0,...,0.0,,0.0,0.0,,0.0,0.0,back/picture/951e1689b3cc41219dc71ea47a2ad237.JPG,0.0,0.0
4,003895c08aa84c2a98d3d1696db7ffcf,cc8cef68e6a14db1bff973219ad73248,003895c08aa84c2a98d3d1696db7ffcf,1,3708822180001080000074,,https://wx.wwsdw.net/sdimg/picture/thumbs/d5a3...,966.0,640.0,0.0,...,127.0,,0.0,0.0,,0.0,0.0,picture/shandongnew/37088221800010/80000074/37...,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20108,ffd22b2ff47d4994b22f1fd7357507b0,23cec260c9b14d3bacae76c27d8db75e,ffd22b2ff47d4994b22f1fd7357507b0,1,3707832180000280000095,,http://www.wwsdw.net/sdimg/picture/thumbs/d55j...,960.0,1440.0,0.0,...,128.0,,0.0,0.0,,0.0,0.0,,0.0,0.0
20109,fff7b77a40a247fb853b651b01c8f0db,f1e5bd76a1074c09a5ec5c32cb880ccc,fff7b77a40a247fb853b651b01c8f0db,3,,00022-A-01.JPG,http://www.wwsdw.net/sdimg/picture/thumbs/a9a6...,3648.0,5472.0,0.0,...,128.0,,0.0,0.0,,0.0,0.0,,0.0,0.0
20110,fff7b77a40a247fb853b651b01c8f0db,3fe7933f349848f08644cd85fea2938c,fff7b77a40a247fb853b651b01c8f0db,3,,00022-D-01.JPG,http://www.wwsdw.net/sdimg/picture/thumbs/030c...,3648.0,5472.0,0.0,...,0.0,,0.0,0.0,,0.0,0.0,,0.0,0.0
20111,fff7b77a40a247fb853b651b01c8f0db,0e8728a2cda04e43a43f6f7b8258cbe6,fff7b77a40a247fb853b651b01c8f0db,3,,00022-F-01.JPG,http://www.wwsdw.net/sdimg/picture/thumbs/5691...,5472.0,3648.0,0.0,...,0.0,,0.0,0.0,,0.0,0.0,,0.0,0.0


In [13]:
# 将空串改为空值；去掉所有空行
df_pics_normalized = df_pics_normalized.replace("", pd.NA)
# df_pics_normalized.dropna(how="all", inplace=True)

# 去掉没有用的行
df_pics_normalized.drop(["thumb4", "thumb4Width", "thumb4Height", "thumb5", "thumb5Width", "thumb5Height", "thumb6Width", "thumb6Height"], axis=1, inplace=True)
df_pics_normalized.drop(['type',"status", "sequence", "createtime"], axis=1, inplace=True)
df_pics_normalized = df_pics_normalized[[
    'id', 'objectId', 'picId','gsNo',    
    'thumb6','picWidth','picHeight',
    'thumb1', 'thumb1Width', 'thumb1Height',
    'url', 'isMain','typeId', 'name',
    'thumb2', 'thumb2Width', 'thumb2Height',
    'thumb3', 'thumb3Width', 'thumb3Height', ]].copy()
df_pics_normalized

,id,objectId,picId,gsNo,thumb6,picWidth,picHeight,thumb1,thumb1Width,thumb1Height,url,isMain,typeId,name,thumb2,thumb2Width,thumb2Height,thumb3,thumb3Width,thumb3Height
0,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,f415bfc279cc411a96d354ec220aecf0,<NA>,back/picture/d40b5bdb48a54875b783d9beefa289d0.JPG,3780.0,2504.0,picture/thumbs/d40b/640x426_d40b5bdb48a54875b7...,640.0,424.0,https://wx.wwsdw.net/sdimg/picture/thumbs/d40b...,2.0,3,987.3.14-A-01.jpg,picture/thumbs/d40b/278x_d40b5bdb48a54875b783d...,278.0,184.0,picture/thumbs/d40b/192x128_d40b5bdb48a54875b7...,192.0,127.0
1,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,85d94a471441437f84fe0f6fa1a88675,<NA>,back/picture/19dfa463937a4960ba3027230ab7301c.JPG,3872.0,2592.0,picture/thumbs/19df/640x426_19dfa463937a4960ba...,0.0,0.0,https://wx.wwsdw.net/sdimg/picture/thumbs/19df...,1.0,3,987.3.14-E-02.jpg,picture/thumbs/19df/278x_19dfa463937a4960ba302...,0.0,0.0,picture/thumbs/19df/192x128_19dfa463937a4960ba...,0.0,0.0
2,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,765f81b3810c475a8888d40062ad9ae3,<NA>,back/picture/20121d4a65444b4ca6a00f69ca531613.JPG,3872.0,2592.0,picture/thumbs/2012/640x426_20121d4a65444b4ca6...,0.0,0.0,https://wx.wwsdw.net/sdimg/picture/thumbs/2012...,1.0,3,987.3.14-E-01.jpg,picture/thumbs/2012/278x_20121d4a65444b4ca6a00...,0.0,0.0,picture/thumbs/2012/192x128_20121d4a65444b4ca6...,0.0,0.0
3,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,9a5ff57108004ae8afe477d03644e1e2,<NA>,back/picture/951e1689b3cc41219dc71ea47a2ad237.JPG,3872.0,2592.0,picture/thumbs/951e/640x426_951e1689b3cc41219d...,0.0,0.0,https://wx.wwsdw.net/sdimg/picture/thumbs/951e...,1.0,3,987.3.14-F-01.jpg,picture/thumbs/951e/278x_951e1689b3cc41219dc71...,0.0,0.0,picture/thumbs/951e/192x128_951e1689b3cc41219d...,0.0,0.0
4,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,cc8cef68e6a14db1bff973219ad73248,3708822180001080000074,picture/shandongnew/37088221800010/80000074/37...,966.0,640.0,picture/thumbs/d5a3do4lm/1bmoga/640x426_370882...,640.0,424.0,https://wx.wwsdw.net/sdimg/picture/thumbs/d5a3...,1.0,1,<NA>,picture/thumbs/d5a3do4lm/1bmoga/278x_370882218...,278.0,184.0,picture/thumbs/d5a3do4lm/1bmoga/192x128_370882...,192.0,127.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20108,ffd22b2ff47d4994b22f1fd7357507b0,ffd22b2ff47d4994b22f1fd7357507b0,23cec260c9b14d3bacae76c27d8db75e,3707832180000280000095,<NA>,960.0,1440.0,picture/thumbs/d55jngrpe/1bmogv/640x426_370783...,284.0,426.0,http://www.wwsdw.net/sdimg/picture/thumbs/d55j...,1.0,1,<NA>,picture/thumbs/d55jngrpe/1bmogv/278x_370783218...,278.0,417.0,picture/thumbs/d55jngrpe/1bmogv/192x128_370783...,85.0,128.0
20109,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,f1e5bd76a1074c09a5ec5c32cb880ccc,<NA>,<NA>,3648.0,5472.0,picture/thumbs/a9a6/640x426_a9a60d035ce343e880...,284.0,426.0,http://www.wwsdw.net/sdimg/picture/thumbs/a9a6...,2.0,3,00022-A-01.JPG,picture/thumbs/a9a6/278x_a9a60d035ce343e8802ee...,278.0,417.0,picture/thumbs/a9a6/192x128_a9a60d035ce343e880...,85.0,128.0
20110,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,3fe7933f349848f08644cd85fea2938c,<NA>,<NA>,3648.0,5472.0,picture/thumbs/030c/640x426_030cc06a872f41669a...,0.0,0.0,http://www.wwsdw.net/sdimg/picture/thumbs/030c...,1.0,3,00022-D-01.JPG,picture/thumbs/030c/278x_030cc06a872f41669a6a8...,0.0,0.0,picture/thumbs/030c/192x128_030cc06a872f41669a...,0.0,0.0
20111,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,0e8728a2cda04e43a43f6f7b8258cbe6,<NA>,<NA>,5472.0,3648.0,picture/thumbs/5691/640x426_5691605d59f543fb87...,0.0,0.0,http://www.wwsdw.net/sdimg/picture/thumbs/5691...,1.0,3,00022-F-01.JPG,picture/thumbs/5691/278x_5691605d59f543fb871f8...,0.0,0.0,picture/thumbs/5691/192x128_5691605d59f543fb87...,0.0,0.0


In [14]:
# 先理解一下缩略图的长宽高，确认是否对于每一行，picWidth和picHeight都比所有缩略图的长和宽更大
df_pics_normalized["size_check"] = df_pics_normalized.apply(
    lambda row: (row["picWidth"] >= max(row["thumb1Width"],row["thumb2Width"],row["thumb3Width"])) and 
                (row["picHeight"] >= max(row["thumb1Height"],row["thumb2Height"],row["thumb3Height"])),
    axis=1
)
mask = df_pics_normalized["size_check"]==False
df_pics_normalized[mask]
# 根据以上代码，可以发现只有25张图的尺寸不满足thumb6比其他都大，可以无视。
# 可以去掉所有thumb2~5的信息，以及所有长宽信息，只保留thumb1和thumb6
df_pics_normalized = df_pics_normalized[[
    'id', 'objectId', 'picId','gsNo',    
    'thumb6','thumb1',
    'url', 'isMain','typeId', 'name',]].copy()

In [15]:
# 有部分行的id和objectId不相等，原因不明，先保留
mask = df_pics_normalized["id"] != df_pics_normalized["objectId"]
df_pics_normalized[mask]

,id,objectId,picId,gsNo,thumb6,thumb1,url,isMain,typeId,name
28,0080D7CDCAED42E99F3186AA3816DD20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,01a7147628c948459d3545a9f8edf830,tempCollection,b461e7e8fb4f4914851a86a80b82e6bc,<NA>,<NA>,back/picture/wenwu/640x426_540a7e25c93548058ea...,http://www.wwsdw.net/sdimg/back/picture/wenwu/...,2.0,3,A (19).JPG
134,023c6c328e1c4ea7b1179289477cd260,tempCollection,74b4412ad1534c45bafbbd7159da454b,<NA>,<NA>,back/picture/wenwu/640x426_aa09cc03705a416a975...,http://www.wwsdw.net/sdimg/back/picture/wenwu/...,2.0,3,0327.png
587,07754cd41d944954aecde1e89aafe727,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1263,0a7301cc87914346889af62cba430588,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
19801,e6c11688896c44f3b26ef817a6f42875,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19899,ed311f0b093f41c9886093942be58f90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20030,f4eb877d688d4a99bb61c63457287895,tempCollection,1abeb0b7b7e24db184ff29df1870b441,<NA>,<NA>,back/picture/wenwu/640x426_3371314650c14c369b1...,http://www.wwsdw.net/sdimg/back/picture/wenwu/...,2.0,3,0336.png
20038,f780f4aab5844d458c1cc35fe5d0398e,tempCollection,df19de32e5e24d3c8df19d501ce4fec9,<NA>,<NA>,back/picture/wenwu/640x426_fca243d75d6a40a0915...,http://www.wwsdw.net/sdimg/back/picture/wenwu/...,2.0,3,0335.png


In [16]:
# 现在分析thumb1和url两列的关系，判断是否thumb1总是url的后缀
def is_thumb1_suffix(row):
    url = row['url']
    thumb1 = row['thumb1']
    if pd.isna(thumb1) or pd.isna(url):
        return False
    return url.endswith(thumb1)
df_pics_normalized["thumb1_is_suffix"] = df_pics_normalized.apply(is_thumb1_suffix, axis=1)
df_pics_normalized["thumb1_is_suffix"].value_counts()
# 运行发现确实全部都是后缀，删除辅助列和thumb1列
df_pics_normalized.drop(["thumb1_is_suffix", "thumb1"], axis=1, inplace=True)

In [17]:
# isMain typeId name这几列都看不出作用，先打包写成一个新列info
def pack_info(row):
    return {
        "isMain": row["isMain"],
        "typeId": row["typeId"],
        "name": row["name"]
    }
df_pics_normalized["info"] = df_pics_normalized.apply(pack_info, axis=1)
df_pics_normalized.drop(["isMain", "typeId", "name"], axis=1, inplace=True)
df_pics_normalized

,id,objectId,picId,gsNo,thumb6,url,info
0,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,f415bfc279cc411a96d354ec220aecf0,<NA>,back/picture/d40b5bdb48a54875b783d9beefa289d0.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/d40b...,"{'isMain': 2.0, 'typeId': '3', 'name': '987.3...."
1,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,85d94a471441437f84fe0f6fa1a88675,<NA>,back/picture/19dfa463937a4960ba3027230ab7301c.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/19df...,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3...."
2,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,765f81b3810c475a8888d40062ad9ae3,<NA>,back/picture/20121d4a65444b4ca6a00f69ca531613.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/2012...,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3...."
3,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,9a5ff57108004ae8afe477d03644e1e2,<NA>,back/picture/951e1689b3cc41219dc71ea47a2ad237.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/951e...,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3...."
4,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,cc8cef68e6a14db1bff973219ad73248,3708822180001080000074,picture/shandongnew/37088221800010/80000074/37...,https://wx.wwsdw.net/sdimg/picture/thumbs/d5a3...,"{'isMain': 1.0, 'typeId': '1', 'name': <NA>}"
...,...,...,...,...,...,...,...
20108,ffd22b2ff47d4994b22f1fd7357507b0,ffd22b2ff47d4994b22f1fd7357507b0,23cec260c9b14d3bacae76c27d8db75e,3707832180000280000095,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/d55j...,"{'isMain': 1.0, 'typeId': '1', 'name': <NA>}"
20109,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,f1e5bd76a1074c09a5ec5c32cb880ccc,<NA>,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/a9a6...,"{'isMain': 2.0, 'typeId': '3', 'name': '00022-..."
20110,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,3fe7933f349848f08644cd85fea2938c,<NA>,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/030c...,"{'isMain': 1.0, 'typeId': '3', 'name': '00022-..."
20111,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,0e8728a2cda04e43a43f6f7b8258cbe6,<NA>,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/5691...,"{'isMain': 1.0, 'typeId': '3', 'name': '00022-..."


In [18]:
df_pics_without_thumb6 = df_pics_normalized[df_pics_normalized["thumb6"].isna()].copy()
df_pics_with_thumb6 = df_pics_normalized[~df_pics_normalized["thumb6"].isna()].copy()

In [19]:
# ==========================================
# 1. 建立可扩展的规则引擎 (策略字典)
# ==========================================
URL_PATTERNS = {
    # 格式1：640x426 的缩略图格式
    # 解析：
    # (\d{14}) 是第1个捕获组，匹配 14 位数字 a
    # (\d{8})  是第2个捕获组，匹配 8 位数字 b
    # \1\2     是反向引用，直接在正则层面要求这里必须等于组1和组2的直接拼接！
    # [A-Z]    匹配一个大写字母
    # \d{4}    匹配4位数字
    "shandong_d4": r"^http://www\.wwsdw\.net/sdimg/picture/shandong/(\d{14})/(\d{8})/thumb/640x426_\1\2-[A-Z]-\d{4}\.jpg$",
    "shandong_d2": r"^http://www\.wwsdw\.net/sdimg/picture/shandong/(\d{14})/(\d{8})/thumb/640x426_\1\2-[A-Z]-\d{2}\.jpg$",
    "shandong_d1": r"^http://www\.wwsdw\.net/sdimg/picture/shandong/(\d{14})/(\d{8})/thumb/640x426_\1\2-[A-Z]-\d{1}\.jpg$",
    "shandongnew_d4": r"^http://www\.wwsdw\.net/sdimg/picture/shandongnew/(\d{14})/(\d{8})/thumb/640x426_\1\2-[A-Z]-\d{4}\.jpg$",
    "back_picture_wenwu": r'^http://www\.wwsdw\.net/sdimg/back/picture/wenwu/640x426_[A-Za-z0-9]+\.(jpg|JPG|PNG)$',
    "back_picture": r'^http://www\.wwsdw\.net/sdimg/back/picture/640x426_[A-Za-z0-9]+\.(jpg|JPG|PNG)$',
    "thumbs_d22_d4":r"^http://www\.wwsdw\.net/sdimg/picture/thumbs/[a-z0-9]+/[a-z0-9]+/640x426_\d{22}-[A-Z]-\d{4}\.(jpg|JPG)$",
    "thumbs_?22_d4":r"^http://www\.wwsdw\.net/sdimg/picture/thumbs/[a-z0-9]+/[a-z0-9]+/640x426_[a-z0-9]{22}-[A-Z]-\d{4}\.jpg$",
    "thumbs_rand":r"^http://www\.wwsdw\.net/sdimg/picture/thumbs/([a-z0-9]+)/640x426_\1[a-z0-9]+.(jpg|JPG|PNG)$",
    # 扩展示例：如果以后你发现了原图格式，只需在这里加一行即可
    # "origin_full": r"^http://www\.wwsdw\.net/sdimg/picture/shandong/(\d{14})/(\d{8})/\1\2-[A-Z]-\d{4}\.jpg$"
}

# ==========================================
# 2. 定义分类器
# ==========================================
def classify_url(url):
    """
    遍历所有的规则，如果匹配上某一条，就返回该规则的名称。
    如果不符合任何规则，返回 'unknown'。
    """
    if pd.isna(url):
        return "empty"
        
    for format_name, pattern in URL_PATTERNS.items():
        if re.match(pattern, url):
            return format_name
            
    return "unknown"

# ==========================================
# 3. 注入标签并进行拆分
# ==========================================
# 生成一列新的分类标签
df_pics_without_thumb6['url_format'] = df_pics_without_thumb6['url'].apply(classify_url)

# 根据标签拆分出你需要的子表格（注意结尾加 .copy() 阻断 SettingWithCopyWarning）
mask = df_pics_without_thumb6['url_format'] != 'unknown'
df_matched = df_pics_without_thumb6[mask].copy()
df_unknown = df_pics_without_thumb6[~mask].copy()

print(f"总数据量: {len(df_pics_without_thumb6)}")
print(f"匹配到格式的数量: {len(df_matched)}")
print(f"不符合现有任何格式的数量: {len(df_unknown)}")

总数据量: 14654
匹配到格式的数量: 14654
不符合现有任何格式的数量: 0


In [20]:
df_pics_without_thumb6['url_format'].value_counts()

url_format
shandong_d4           11666
shandongnew_d4         1429
thumbs_d22_d4          1099
thumbs_rand             313
back_picture_wenwu       80
empty                    27
shandong_d1              27
shandong_d2               7
thumbs_?22_d4             5
back_picture              1
Name: count, dtype: int64

In [21]:
for format_name in df_pics_without_thumb6['url_format'].unique():
    if format_name == "empty":
         continue
    if format_name == "back_picture":
        i = 1
    else:
        i = 5
    example_urls = df_pics_without_thumb6[df_pics_without_thumb6['url_format'] == format_name]['url'].sample(i).tolist()
    print(f"\n格式: {format_name} | 样例 URL:")
    for url in example_urls:
        print(url)


格式: shandong_d4 | 样例 URL:
http://www.wwsdw.net/sdimg/picture/shandong/37028121800001/00001124/thumb/640x426_3702812180000100001124-B-0001.jpg
http://www.wwsdw.net/sdimg/picture/shandong/37040221800003/00006322/thumb/640x426_3704022180000300006322-C-0004.jpg
http://www.wwsdw.net/sdimg/picture/shandong/37040221800003/00007357/thumb/640x426_3704022180000300007357-A-0001.jpg
http://www.wwsdw.net/sdimg/picture/shandong/37040221800003/00004395/thumb/640x426_3704022180000300004395-C-0003.jpg
http://www.wwsdw.net/sdimg/picture/shandong/37032121800004/00003109/thumb/640x426_3703212180000400003109-B-0001.jpg

格式: shandongnew_d4 | 样例 URL:
http://www.wwsdw.net/sdimg/picture/shandongnew/37072421800002/00096503/thumb/640x426_3707242180000200096503-B-0001.jpg
http://www.wwsdw.net/sdimg/picture/shandongnew/37061221800002/00003380/thumb/640x426_3706122180000200003380-A-0001.jpg
http://www.wwsdw.net/sdimg/picture/shandongnew/37132521800002/00012143/thumb/640x426_3713252180000200012143-C-0001.jpg
http:/

In [22]:


# # ==========================================
# # 1. 建立可扩展的规则引擎 (策略字典)
# # ==========================================
# URL_PATTERNS_THUMB6 = {
#     # 格式1：640x426 的缩略图格式
#     # 解析：
#     # (\d{14}) 是第1个捕获组，匹配 14 位数字 a
#     # (\d{8})  是第2个捕获组，匹配 8 位数字 b
#     # \1\2     是反向引用，直接在正则层面要求这里必须等于组1和组2的直接拼接！
#     # [A-Z]    匹配一个大写字母
#     # \d{4}    匹配4位数字
#     "shandong_d4": r"^picture/shandong/(\d{14})/(\d{8})/\1\2-[A-Z]-\d{4}\.jpg$",
#     "shandong_d2": r"^picture/shandong/(\d{14})/(\d{8})/\1\2-[A-Z]-\d{2}\.jpg$",
#     "shandong_d1": r"^picture/shandong/(\d{14})/(\d{8})/\1\2-[a-zA-Z]-\d{1}\.(jpg|jpg\.jpg)$",
#     "shandongnew_d4": r"^picture/shandongnew/(\d{14})/(\d{8})/\1\2-[A-Z]-\d{4}\.(jpg|JPG)$",
#     "shandongnew_d1": r"^picture/shandongnew/(\d{14})/(\d{8})/\1\2-[a-zA-Z]-\d{1}\.(jpg|JPG)$",
#     "back_picture": r'^back/picture/[A-Za-z0-9]+\.(jpg|JPG|PNG)$',
#     # 扩展示例：如果以后你发现了原图格式，只需在这里加一行即可
#     # "origin_full": r"^http://www\.wwsdw\.net/sdimg/picture/shandong/(\d{14})/(\d{8})/\1\2-[A-Z]-\d{4}\.jpg$"
# }

# # ==========================================
# # 2. 定义分类器
# # ==========================================
# def classify_url_thumb6(url):
#     """
#     遍历所有的规则，如果匹配上某一条，就返回该规则的名称。
#     如果不符合任何规则，返回 'unknown'。
#     """
#     if pd.isna(url):
#         return "empty"
        
#     for format_name, pattern in URL_PATTERNS_THUMB6.items():
#         if re.match(pattern, url):
#             return format_name
            
#     return "unknown"

# # ==========================================
# # 3. 注入标签并进行拆分
# # ==========================================
# # 生成一列新的分类标签
# df_pics_with_thumb6['url_format'] = df_pics_with_thumb6['thumb6'].apply(classify_url_thumb6)

# # 根据标签拆分出你需要的子表格（注意结尾加 .copy() 阻断 SettingWithCopyWarning）
# mask = df_pics_with_thumb6['url_format'] != 'unknown'
# df_highres_matched = df_pics_with_thumb6[mask].copy()
# df_highres_unknown = df_pics_with_thumb6[~mask].copy()

# print(f"总数据量: {len(df_pics_with_thumb6)}")
# print(f"匹配到格式的数量: {len(df_highres_matched)}")
# print(f"不符合现有任何格式的数量: {len(df_highres_unknown)}")
# mask = df_pics_with_thumb6['url_format'] == "unknown"
# df_pics_with_thumb6[mask]

In [23]:
# 猜测的高清url规则：
# 对于shandong_d4、shandong_d2、shandong_d1、shandongnew_d4，猜测只需要把其中的"thumb/640x426_"去掉
#     "shandong_d4": r"^http://www\.wwsdw\.net/sdimg/picture/shandong/(\d{14})/(\d{8})/thumb/640x426_\1\2-[A-Z]-\d{4}\.jpg$",
#     "shandong_d2": r"^http://www\.wwsdw\.net/sdimg/picture/shandong/(\d{14})/(\d{8})/thumb/640x426_\1\2-[A-Z]-\d{2}\.jpg$",
#     "shandong_d1": r"^http://www\.wwsdw\.net/sdimg/picture/shandong/(\d{14})/(\d{8})/thumb/640x426_\1\2-[A-Z]-\d{1}\.jpg$",
#     "shandongnew_d4": r"^http://www\.wwsdw\.net/sdimg/picture/shandongnew/(\d{14})/(\d{8})/thumb/640x426_\1\2-[A-Z]-\d{4}\.jpg$",
# 对于back_picture_wenwu和back_picture，猜测只需要把"640x426_"去掉
#     "back_picture_wenwu": r'^http://www\.wwsdw\.net/sdimg/back/picture/wenwu/640x426_[A-Za-z0-9]+\.(jpg|JPG|PNG)$',
#     "back_picture": r'^http://www\.wwsdw\.net/sdimg/back/picture/640x426_[A-Za-z0-9]+\.(jpg|JPG|PNG)$',
# 对于thumbs_d22_d4和thumbs_?22_d4，猜测需要利用文件名中的22位字母或数字信息，拼凑出类似shandong_d4的形式。具体而言，设url中的那个22位字母或数字串的前14位为stra，后8位为strb，则高清url格式可能为f"http://www.wwsdw.net/sdimg/picture/shandong/{stra}/{strb}/{stra+strb}-X-xxxx.jpg"，另外shandong也可能是shandongnew
#     "thumbs_d22_d4":r"^http://www\.wwsdw\.net/sdimg/picture/thumbs/[a-z0-9]+/[a-z0-9]+/640x426_\d{22}-[A-Z]-\d{4}\.(jpg|JPG)$",
#     "thumbs_?22_d4":r"^http://www\.wwsdw\.net/sdimg/picture/thumbs/[a-z0-9]+/[a-z0-9]+/640x426_[a-z0-9]{22}-[A-Z]-\d{4}\.jpg$",
# 对于thumbs_rand，目前暂时实在没啥办法，猜不出来，只能后续再处理
# 更新：猜测需要在/picture前加上/back，然后删掉从/thumbs到/640x426_之间的部分，即f"http://www.wwsdw.net/sdimg/back/picture/xxxx.jpg"
#     "thumbs_rand":r"^http://www\.wwsdw\.net/sdimg/picture/thumbs/([a-z0-9]+)/640x426_\1[a-z0-9]+.(jpg|JPG|PNG)$",

In [24]:


def generate_highres_url(row):
    """
    根据 url_format 分类，推测高清图的 URL。
    返回单个字符串（确定的URL），列表（多个可能的URL），或 pd.NA（无法推测）。
    """
    url = row['url']
    fmt = row['url_format']
    
    # 异常值防御
    if pd.isna(url) or pd.isna(fmt):
        return pd.NA
        
    # =======================================================
    # 规则组 1：直接去掉 "thumb/640x426_"
    # =======================================================
    if fmt in ["shandong_d4", "shandong_d2", "shandong_d1", "shandongnew_d4"]:
        return url.replace("thumb/640x426_", "")
        
    # =======================================================
    # 规则组 2：直接去掉 "640x426_"
    # =======================================================
    elif fmt in ["back_picture_wenwu", "back_picture"]:
        return url.replace("640x426_", "")
        
    # =======================================================
    # 规则组 3：提取 22 位特征码进行重组，返回待选列表
    # =======================================================
    elif fmt in ["thumbs_d22_d4", "thumbs_?22_d4"]:
        # 提取 "640x426_" 之后的所有内容作为实际文件名
        # 例：1234567890123412345678-A-0001.jpg
        parts = url.split("640x426_")
        if len(parts) > 1:
            filename = parts[-1]
            
            # 确保文件名长度足够提取 22 位
            if len(filename) >= 22:
                stra = filename[:14]   # 前 14 位
                strb = filename[14:22] # 后 8 位
                
                # 构造两种可能的高清路径
                url_shandong = f"http://www.wwsdw.net/sdimg/picture/shandong/{stra}/{strb}/{filename}"
                url_shandongnew = f"http://www.wwsdw.net/sdimg/picture/shandongnew/{stra}/{strb}/{filename}"
                
                return [url_shandongnew, url_shandong]
                
        return pd.NA
        
    # =======================================================
    # 规则组 4：无法推测 (如 thumbs_rand 等)
    # =======================================================
    else:
        parts = url.split("640x426_")
        if len(parts) > 1:
            filename = parts[-1]
            url_back = f"http://www.wwsdw.net/sdimg/back/picture/{filename}"
            return url_back

# 为了防止之前的切片操作引发 SettingWithCopyWarning，养成好习惯，确保当前表是独立的
# 如果你确信它已经是独立的，这行 copy() 也可以省略，但加上更安全
df_pics_without_thumb6 = df_pics_without_thumb6.copy()

# 应用推测逻辑，生成新列
df_pics_without_thumb6['highres_url_guess'] = df_pics_without_thumb6.apply(generate_highres_url, axis=1)

# 打印各类型的推测覆盖率情况
print("高清 URL 推测结果统计：")
print(df_pics_without_thumb6['highres_url_guess'].notna().value_counts())

# 抽查
print("\n--- 抽查 thumbs_rand 类型的推测结果 ---")
sample_d22 = df_pics_without_thumb6[df_pics_without_thumb6['url_format'] == 'thumbs_rand'].sample(2)
for idx, row in sample_d22.iterrows():
    print(f"原缩略图: {row['url']}")
    print(f"推测列表: {row['highres_url_guess']}\n")

高清 URL 推测结果统计：
highres_url_guess
True     14627
False       27
Name: count, dtype: int64

--- 抽查 thumbs_rand 类型的推测结果 ---
原缩略图: http://www.wwsdw.net/sdimg/picture/thumbs/2000/640x426_20003b6048fc46908dc6c53eaecedddc.JPG
推测列表: http://www.wwsdw.net/sdimg/back/picture/20003b6048fc46908dc6c53eaecedddc.JPG

原缩略图: http://www.wwsdw.net/sdimg/picture/thumbs/3a05/640x426_3a0511c2d2ef4a30b2c440274c7e4e6c.JPG
推测列表: http://www.wwsdw.net/sdimg/back/picture/3a0511c2d2ef4a30b2c440274c7e4e6c.JPG



In [25]:
# 对于df_pics_with_thumb6的每一行：在thumb6列的url字符串前面拼接上"http://www.wwsdw.net/sdimg/"，形成完整的图片url，并直接下载到IMG_DOWNLOAD_DIR中，文件名使用picId列的值。
# 对于df_pics_without_thumb6的每一行：如果highres_url_guess列的值是一个字符串，就直接下载这个url指向的图片；如果是一个列表，就依次尝试下载列表中的每个url，直到遇到第一个下载成功的；如果是空值，就跳过不处理（但也应记录在日志中）。下载时，文件名使用picId列的值。
# 下载代码应可以重复运行，这意味着下载前需要扫描下载文件夹，并和表格中的picId核对，只下载那些picId不存在于下载文件夹中的行。
# 日志记录在LOG_DIR文件夹中，日志文件名应带有当前时间戳

In [26]:
# ==========================================
# 1. 路径与全局配置
# ==========================================
IMG_DOWNLOAD_DIR = BASE_DIR / "highres_images"

# 设置带有时间戳的日志文件
current_time = time.strftime("%Y%m%d_%H%M%S")
LOG_FILE = LOG_DIR / f"download_images_{current_time}.log"

# # ==========================================
# # 2. 日志系统配置
# # ==========================================
# # 每次运行重新配置 logger，避免多次运行单元格导致日志重复输出
# for handler in logging.root.handlers[:]:
#     logging.root.removeHandler(handler)

# logging.basicConfig(
#     level=logging.INFO,
#     format='%(asctime)s - %(levelname)s - %(message)s',
#     handlers=[
#         logging.FileHandler(LOG_FILE, encoding='utf-8')
#     ]
# )
# logger = logging.getLogger(__name__)

# # ==========================================
# # 3. 核心下载逻辑
# # ==========================================
# def download_single_image(pic_id, url_list):
#     """
#     尝试下载列表中的 URL，成功一个即返回。
#     返回: (pic_id, 是否成功, 最终使用的URL, 错误信息)
#     """
#     headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
#     last_error = "No URLs to try"
    
#     for url in url_list:
#         try:
#             # 提取文件后缀 (例如从 .jpg, .JPG, .png 中提取)
#             # 如果 URL 中没有明显后缀，默认使用 .jpg
#             ext = url.split('.')[-1]
#             # if len(ext) > 4 or "/" in ext: 
#             #     ext = "jpg"
                
#             file_path = IMG_DOWNLOAD_DIR / f"{pic_id}.{ext}"
            
#             # 发起请求 (stream=True 适合下载大文件，防止内存溢出)
#             response = requests.get(url, headers=headers, stream=True, timeout=15)
#             response.raise_for_status()
            
#             # 分块写入文件
#             with open(file_path, "wb") as f:
#                 for chunk in response.iter_content(chunk_size=8192):
#                     f.write(chunk)
                    
#             return str(pic_id), True, url, None
            
#         except Exception as e:
#             last_error = f"URL: {url} | Error: {str(e)}"
#             continue # 当前 URL 失败，继续尝试列表中的下一个
            
#     # 如果循环结束都没成功
#     return str(pic_id), False, None, last_error

# # ==========================================
# # 4. 任务构建与状态恢复
# # ==========================================
# # 获取已下载的文件集合（仅比对文件名主体，忽略后缀）
# # # 例如 "12345.jpg" -> "12345"
# downloaded_ids = {p.stem for p in IMG_DOWNLOAD_DIR.glob('*') if p.is_file()}
# logger.info(f"状态恢复：本地已存在 {len(downloaded_ids)} 张图片。")

# pending_tasks = [] # 格式: [{'pic_id': '...', 'urls': [...]}]

# # --- 处理 df_pics_with_thumb6 ---
# for _, row in df_pics_with_thumb6.iterrows():
#     pic_id = str(row['picId'])
#     thumb6 = row['thumb6']
    
#     if pic_id in downloaded_ids:
#         continue
        
#     if pd.isna(thumb6):
#         logger.warning(f"跳过 | ID: {pic_id} | 原因: thumb6 为空")
#         continue
        
#     full_url = f"http://www.wwsdw.net/sdimg/{thumb6}"
#     pending_tasks.append({'pic_id': pic_id, 'urls': [full_url]})

# # --- 处理 df_pics_without_thumb6 ---
# for _, row in df_pics_without_thumb6.iterrows():
#     pic_id = str(row['picId'])
#     guess = row['highres_url_guess']
    
#     if pic_id in downloaded_ids:
#         continue
        
#     if not isinstance(guess,list) and pd.isna(guess):   # 如果是列表就可以保证进入后续处理，如果不是列表，再判断是否是空值，如果是空值就拦截
#         logger.warning(f"跳过 | ID: {pic_id} | 原因: 无法推测高清 URL (空值)")
#         continue
        
#     # 将单个字符串和列表统一转化为列表格式，方便下载函数遍历
#     urls_to_try = [guess] if isinstance(guess, str) else guess
#     pending_tasks.append({'pic_id': pic_id, 'urls': urls_to_try})

# # ==========================================
# # 5. 并发执行
# # ==========================================
# if not pending_tasks:
#     print("🎉 扫描完毕：所有图片均已下载，无需执行任务！")
# else:
#     print(f"准备下载：共发现 {len(pending_tasks)} 个待下载任务。")
#     logger.info(f"开始执行下载，待处理数量: {len(pending_tasks)}")
    
#     success_count = 0
#     fail_count = 0
#     max_workers = 10 # 下载图片受限于带宽，线程数不需要开太大，10~15足够
    
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         # 提交所有任务
#         future_to_id = {
#             executor.submit(download_single_image, task['pic_id'], task['urls']): task['pic_id'] 
#             for task in pending_tasks
#         }
        
#         # 实时追踪进度
#         for future in tqdm(as_completed(future_to_id), total=len(pending_tasks), desc="图片下载进度"):
#             pic_id = future_to_id[future]
#             try:
#                 pid, success, final_url, error_msg = future.result()
#                 if success:
#                     success_count += 1
#                     logger.info(f"成功 | ID: {pid} | URL: {final_url}")
#                 else:
#                     fail_count += 1
#                     logger.error(f"失败 | ID: {pid} | 最终错误: {error_msg}")
#             except Exception as e:
#                 fail_count += 1
#                 logger.error(f"线程崩溃 | ID: {pic_id} | 错误: {str(e)}")

#     print(f"\n本轮任务结束。成功下载: {success_count} 张，失败: {fail_count} 张。")
#     print(f"日志已保存至: {LOG_FILE}")
#     if fail_count > 0:
#         print("⚠️ 存在失败请求。直接重新运行当前单元格即可进行重试（已下载的将被自动跳过）。")

In [27]:
# 现在只需要将这些图片整理回去。
# 首先将df_pics_with_thumb6和df_pics_without_thumb6重新拼起来。
df_pics_final = pd.concat([df_pics_with_thumb6, df_pics_without_thumb6], ignore_index=True)
df_pics_final

,id,objectId,picId,gsNo,thumb6,url,info,url_format,highres_url_guess
0,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,f415bfc279cc411a96d354ec220aecf0,<NA>,back/picture/d40b5bdb48a54875b783d9beefa289d0.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/d40b...,"{'isMain': 2.0, 'typeId': '3', 'name': '987.3....",NaN,NaN
1,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,85d94a471441437f84fe0f6fa1a88675,<NA>,back/picture/19dfa463937a4960ba3027230ab7301c.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/19df...,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3....",NaN,NaN
2,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,765f81b3810c475a8888d40062ad9ae3,<NA>,back/picture/20121d4a65444b4ca6a00f69ca531613.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/2012...,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3....",NaN,NaN
3,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,9a5ff57108004ae8afe477d03644e1e2,<NA>,back/picture/951e1689b3cc41219dc71ea47a2ad237.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/951e...,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3....",NaN,NaN
4,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,cc8cef68e6a14db1bff973219ad73248,3708822180001080000074,picture/shandongnew/37088221800010/80000074/37...,https://wx.wwsdw.net/sdimg/picture/thumbs/d5a3...,"{'isMain': 1.0, 'typeId': '1', 'name': <NA>}",NaN,NaN
...,...,...,...,...,...,...,...,...,...
20108,ffd22b2ff47d4994b22f1fd7357507b0,ffd22b2ff47d4994b22f1fd7357507b0,23cec260c9b14d3bacae76c27d8db75e,3707832180000280000095,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/d55j...,"{'isMain': 1.0, 'typeId': '1', 'name': <NA>}",thumbs_d22_d4,[http://www.wwsdw.net/sdimg/picture/shandongne...
20109,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,f1e5bd76a1074c09a5ec5c32cb880ccc,<NA>,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/a9a6...,"{'isMain': 2.0, 'typeId': '3', 'name': '00022-...",thumbs_rand,http://www.wwsdw.net/sdimg/back/picture/a9a60d...
20110,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,3fe7933f349848f08644cd85fea2938c,<NA>,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/030c...,"{'isMain': 1.0, 'typeId': '3', 'name': '00022-...",thumbs_rand,http://www.wwsdw.net/sdimg/back/picture/030cc0...
20111,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,0e8728a2cda04e43a43f6f7b8258cbe6,<NA>,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/5691...,"{'isMain': 1.0, 'typeId': '3', 'name': '00022-...",thumbs_rand,http://www.wwsdw.net/sdimg/back/picture/569160...


In [28]:
# 将日志文件中的url登记到表格中，形成一列新的实际下载URL
# 日志文件包括LOG_DIR中以"download_images_"开头的两个文件
# 日志文件中的每一行都是类似于以下格式的文本：
# 2024-06-01 12:00:00,000 - INFO - 成功 | ID: 12345 | URL: http://www.wwsdw.net/xxx/xxx/xxx/xxx.jpg
# 只需将日志文件中每行的ID和URL提取出来，形成一个字典{id: url}，然后用这个字典去更新df_pics_final表格中的一列新列actual_url，用每行的picId去字典里查询url并填入actual_url列，如果某行的picId在字典里没有对应的url，就填入空值。
log_files = list(LOG_DIR.glob("download_images_*.log"))
downloaded_url_dict = {}
for log_file in log_files:
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if "成功 | ID:" in line and "| URL: " in line:
                try:
                    # 提取ID和URL
                    id_part = line.split("成功 | ID:")[1].split("| URL:")[0].strip()
                    url_part = line.split("| URL:")[1].strip()
                    downloaded_url_dict[id_part] = url_part
                except Exception as e:
                    print(f"日志解析错误 | 行内容: {line} | 错误: {str(e)}")

df_pics_final['actual_url'] = df_pics_final['picId'].apply(lambda x: downloaded_url_dict.get(str(x), pd.NA))
df_pics_final

,id,objectId,picId,gsNo,thumb6,url,info,url_format,highres_url_guess,actual_url
0,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,f415bfc279cc411a96d354ec220aecf0,<NA>,back/picture/d40b5bdb48a54875b783d9beefa289d0.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/d40b...,"{'isMain': 2.0, 'typeId': '3', 'name': '987.3....",NaN,NaN,http://www.wwsdw.net/sdimg/back/picture/d40b5b...
1,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,85d94a471441437f84fe0f6fa1a88675,<NA>,back/picture/19dfa463937a4960ba3027230ab7301c.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/19df...,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3....",NaN,NaN,http://www.wwsdw.net/sdimg/back/picture/19dfa4...
2,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,765f81b3810c475a8888d40062ad9ae3,<NA>,back/picture/20121d4a65444b4ca6a00f69ca531613.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/2012...,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3....",NaN,NaN,http://www.wwsdw.net/sdimg/back/picture/20121d...
3,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,9a5ff57108004ae8afe477d03644e1e2,<NA>,back/picture/951e1689b3cc41219dc71ea47a2ad237.JPG,https://wx.wwsdw.net/sdimg/picture/thumbs/951e...,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3....",NaN,NaN,http://www.wwsdw.net/sdimg/back/picture/951e16...
4,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,cc8cef68e6a14db1bff973219ad73248,3708822180001080000074,picture/shandongnew/37088221800010/80000074/37...,https://wx.wwsdw.net/sdimg/picture/thumbs/d5a3...,"{'isMain': 1.0, 'typeId': '1', 'name': <NA>}",NaN,NaN,http://www.wwsdw.net/sdimg/picture/shandongnew...
...,...,...,...,...,...,...,...,...,...,...
20108,ffd22b2ff47d4994b22f1fd7357507b0,ffd22b2ff47d4994b22f1fd7357507b0,23cec260c9b14d3bacae76c27d8db75e,3707832180000280000095,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/d55j...,"{'isMain': 1.0, 'typeId': '1', 'name': <NA>}",thumbs_d22_d4,[http://www.wwsdw.net/sdimg/picture/shandongne...,http://www.wwsdw.net/sdimg/picture/shandongnew...
20109,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,f1e5bd76a1074c09a5ec5c32cb880ccc,<NA>,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/a9a6...,"{'isMain': 2.0, 'typeId': '3', 'name': '00022-...",thumbs_rand,http://www.wwsdw.net/sdimg/back/picture/a9a60d...,http://www.wwsdw.net/sdimg/back/picture/a9a60d...
20110,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,3fe7933f349848f08644cd85fea2938c,<NA>,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/030c...,"{'isMain': 1.0, 'typeId': '3', 'name': '00022-...",thumbs_rand,http://www.wwsdw.net/sdimg/back/picture/030cc0...,http://www.wwsdw.net/sdimg/back/picture/030cc0...
20111,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,0e8728a2cda04e43a43f6f7b8258cbe6,<NA>,<NA>,http://www.wwsdw.net/sdimg/picture/thumbs/5691...,"{'isMain': 1.0, 'typeId': '3', 'name': '00022-...",thumbs_rand,http://www.wwsdw.net/sdimg/back/picture/569160...,http://www.wwsdw.net/sdimg/back/picture/569160...


In [29]:
# 当前df_pics_final的各列：
# 'id', 'objectId'：用于和之前的数据对接，保留
# 'picId'：图片本身的唯一标识，保留
# 'gsNo'：可以考虑加入info当中去
# 'info'：保留
# 'thumb6', 'url', 'actual_url'：thumb6已经被整合进acturl_url，url用于推导actual_url，只有那些推导不出actual_url的行才有必要保留url
# 'url_format', 'highres_url_guess'：前者是归纳出的格式，后者是根据格式进行的猜测，都是中间产物，实际使用到的已经被完全记录在actual_url,可以删除
df_pics_final = (
    df_pics_final
    # 1. 向量化判断，生成新列
    # 注意这里使用了 lambda df: df... ，df 代表经过前面的操作后此刻的状态（虽然这里是第一步）
    .assign(got_highres=lambda df: df['actual_url'].notna())
    
    # 2. 向量化条件填充，替代缓慢的 apply if-else
    # fillna 可以完美实现：“如果有值就保留，如果是空值就用参数里的序列对应位置的值填补”
    .assign(actual_url=lambda df: df['actual_url'].fillna(df['url']))
    
    # 3. 更新字典。必须用 apply，但要包裹在最外层的 lambda df: 里面，以获取最新的 got_highres
    .assign(info=lambda df: df.apply(
        lambda row: {**row["info"], "gsNo": row["gsNo"], "highRes": row["got_highres"]}, 
        axis=1
    ))
    
    # 4. 收尾清理
    # drop 推荐明确使用 columns= 参数，语义更清晰，不需要写 axis=1
    .drop(columns=['thumb6', 'url_format', 'url', 'highres_url_guess', "gsNo", "got_highres"])
    .rename(columns={"actual_url": "online_url"})
)
df_pics_final

,id,objectId,picId,info,online_url
0,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,f415bfc279cc411a96d354ec220aecf0,"{'isMain': 2.0, 'typeId': '3', 'name': '987.3....",http://www.wwsdw.net/sdimg/back/picture/d40b5b...
1,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,85d94a471441437f84fe0f6fa1a88675,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3....",http://www.wwsdw.net/sdimg/back/picture/19dfa4...
2,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,765f81b3810c475a8888d40062ad9ae3,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3....",http://www.wwsdw.net/sdimg/back/picture/20121d...
3,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,9a5ff57108004ae8afe477d03644e1e2,"{'isMain': 1.0, 'typeId': '3', 'name': '987.3....",http://www.wwsdw.net/sdimg/back/picture/951e16...
4,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,cc8cef68e6a14db1bff973219ad73248,"{'isMain': 1.0, 'typeId': '1', 'name': <NA>, '...",http://www.wwsdw.net/sdimg/picture/shandongnew...
...,...,...,...,...,...
20108,ffd22b2ff47d4994b22f1fd7357507b0,ffd22b2ff47d4994b22f1fd7357507b0,23cec260c9b14d3bacae76c27d8db75e,"{'isMain': 1.0, 'typeId': '1', 'name': <NA>, '...",http://www.wwsdw.net/sdimg/picture/shandongnew...
20109,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,f1e5bd76a1074c09a5ec5c32cb880ccc,"{'isMain': 2.0, 'typeId': '3', 'name': '00022-...",http://www.wwsdw.net/sdimg/back/picture/a9a60d...
20110,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,3fe7933f349848f08644cd85fea2938c,"{'isMain': 1.0, 'typeId': '3', 'name': '00022-...",http://www.wwsdw.net/sdimg/back/picture/030cc0...
20111,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,0e8728a2cda04e43a43f6f7b8258cbe6,"{'isMain': 1.0, 'typeId': '3', 'name': '00022-...",http://www.wwsdw.net/sdimg/back/picture/569160...


In [30]:
# 检查可发现id与objectId不等的情况只出现于objectId为“tempCollection”的情况，这些文物的数据和详情页并没有真正的问题。除此之外就是真的没有图（故爬不到objectId）。前者可以保留，后者相当于完全的空行，不影响后续的拼接。
# df_pics_final.query("id != objectId")

In [31]:
# 最终数据的images列中，每张图需要记录：id、url、extension、name、description
# 对于每张图，我们已有的信息：id、objectId、picId、info、online_url
# id：直接用picId
# url：这里其实指的是图片本地路径的url，需要查询并拼接一下
# extension：可以从online_url中提取
# name：info中或许有名字？也可以不填
# description：可以考虑不填

In [32]:
df_pics_final = (
    df_pics_final
    .assign(extension=lambda df: df['online_url'].str.split('.').str[-1])
    .assign(name=lambda df: df['info'].str.get('name').replace({np.nan: None}))
    .assign(description=None)
    .drop(columns=['info'])
)
df_pics_final

,id,objectId,picId,online_url,extension,name,description
0,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,f415bfc279cc411a96d354ec220aecf0,http://www.wwsdw.net/sdimg/back/picture/d40b5b...,JPG,987.3.14-A-01.jpg,None
1,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,85d94a471441437f84fe0f6fa1a88675,http://www.wwsdw.net/sdimg/back/picture/19dfa4...,JPG,987.3.14-E-02.jpg,None
2,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,765f81b3810c475a8888d40062ad9ae3,http://www.wwsdw.net/sdimg/back/picture/20121d...,JPG,987.3.14-E-01.jpg,None
3,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,9a5ff57108004ae8afe477d03644e1e2,http://www.wwsdw.net/sdimg/back/picture/951e16...,JPG,987.3.14-F-01.jpg,None
4,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,cc8cef68e6a14db1bff973219ad73248,http://www.wwsdw.net/sdimg/picture/shandongnew...,jpg,None,None
...,...,...,...,...,...,...,...
20108,ffd22b2ff47d4994b22f1fd7357507b0,ffd22b2ff47d4994b22f1fd7357507b0,23cec260c9b14d3bacae76c27d8db75e,http://www.wwsdw.net/sdimg/picture/shandongnew...,jpg,None,None
20109,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,f1e5bd76a1074c09a5ec5c32cb880ccc,http://www.wwsdw.net/sdimg/back/picture/a9a60d...,JPG,00022-A-01.JPG,None
20110,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,3fe7933f349848f08644cd85fea2938c,http://www.wwsdw.net/sdimg/back/picture/030cc0...,JPG,00022-D-01.JPG,None
20111,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,0e8728a2cda04e43a43f6f7b8258cbe6,http://www.wwsdw.net/sdimg/back/picture/569160...,JPG,00022-F-01.JPG,None


In [33]:
# 需要拼接出每行图片的本地路径，填入local_url
# 对于每个picId非空的行，只需获取其extension列
# 然后填写"data/images/wwsd/{picId}.{extension}"
df_pics_final = (
    df_pics_final
    .assign(
        # 第一步：不管 picId 是否为空，先粗暴地进行全列的向量化拼接
        # Series 之间的加号 (+) 在底层是用 C 语言执行的字符串拼接，速度极快
        local_url=lambda df: "data/images/wwsd/" + df['picId'].astype(str) + "." + df['extension'].astype(str)
    )
    .assign(
        # 第二步：安全兜底
        # 因为上一步中，如果 picId 是 NaN，astype(str) 会把它变成字符串 "nan" 拼进去。
        # 所以我们用 np.where 盖上一层“遮罩”，把原本 picId 为空的行强行抹除置为 pd.NA
        local_url=lambda df: np.where(df['picId'].isna(), pd.NA, df['local_url'])
    )
)
df_pics_final

,id,objectId,picId,online_url,extension,name,description,local_url
0,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,f415bfc279cc411a96d354ec220aecf0,http://www.wwsdw.net/sdimg/back/picture/d40b5b...,JPG,987.3.14-A-01.jpg,None,data/images/wwsd/f415bfc279cc411a96d354ec220ae...
1,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,85d94a471441437f84fe0f6fa1a88675,http://www.wwsdw.net/sdimg/back/picture/19dfa4...,JPG,987.3.14-E-02.jpg,None,data/images/wwsd/85d94a471441437f84fe0f6fa1a88...
2,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,765f81b3810c475a8888d40062ad9ae3,http://www.wwsdw.net/sdimg/back/picture/20121d...,JPG,987.3.14-E-01.jpg,None,data/images/wwsd/765f81b3810c475a8888d40062ad9...
3,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,9a5ff57108004ae8afe477d03644e1e2,http://www.wwsdw.net/sdimg/back/picture/951e16...,JPG,987.3.14-F-01.jpg,None,data/images/wwsd/9a5ff57108004ae8afe477d03644e...
4,003895c08aa84c2a98d3d1696db7ffcf,003895c08aa84c2a98d3d1696db7ffcf,cc8cef68e6a14db1bff973219ad73248,http://www.wwsdw.net/sdimg/picture/shandongnew...,jpg,None,None,data/images/wwsd/cc8cef68e6a14db1bff973219ad73...
...,...,...,...,...,...,...,...,...
20108,ffd22b2ff47d4994b22f1fd7357507b0,ffd22b2ff47d4994b22f1fd7357507b0,23cec260c9b14d3bacae76c27d8db75e,http://www.wwsdw.net/sdimg/picture/shandongnew...,jpg,None,None,data/images/wwsd/23cec260c9b14d3bacae76c27d8db...
20109,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,f1e5bd76a1074c09a5ec5c32cb880ccc,http://www.wwsdw.net/sdimg/back/picture/a9a60d...,JPG,00022-A-01.JPG,None,data/images/wwsd/f1e5bd76a1074c09a5ec5c32cb880...
20110,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,3fe7933f349848f08644cd85fea2938c,http://www.wwsdw.net/sdimg/back/picture/030cc0...,JPG,00022-D-01.JPG,None,data/images/wwsd/3fe7933f349848f08644cd85fea29...
20111,fff7b77a40a247fb853b651b01c8f0db,fff7b77a40a247fb853b651b01c8f0db,0e8728a2cda04e43a43f6f7b8258cbe6,http://www.wwsdw.net/sdimg/back/picture/569160...,JPG,00022-F-01.JPG,None,data/images/wwsd/0e8728a2cda04e43a43f6f7b8258c...


In [34]:
# 现在将每张图整理成一个字典，放在final_info列中
# 最终整理之前，应该把那些picId为空的行全都删掉！
df_pics_final = df_pics_final[~df_pics_final['picId'].isna()].copy()
def create_final_info(row):
    return {
        "id": row["picId"],
        "url": row["local_url"],
        "extension": row["extension"],
        "name": row["name"],
        "description": row["description"],
    }
df_pics_final['final_info'] = df_pics_final.apply(create_final_info, axis=1)
df_pics_to_group = df_pics_final[["id", "final_info"]].copy()
df_pics_to_group

,id,final_info
0,003895c08aa84c2a98d3d1696db7ffcf,"{'id': 'f415bfc279cc411a96d354ec220aecf0', 'ur..."
1,003895c08aa84c2a98d3d1696db7ffcf,"{'id': '85d94a471441437f84fe0f6fa1a88675', 'ur..."
2,003895c08aa84c2a98d3d1696db7ffcf,"{'id': '765f81b3810c475a8888d40062ad9ae3', 'ur..."
3,003895c08aa84c2a98d3d1696db7ffcf,"{'id': '9a5ff57108004ae8afe477d03644e1e2', 'ur..."
4,003895c08aa84c2a98d3d1696db7ffcf,"{'id': 'cc8cef68e6a14db1bff973219ad73248', 'ur..."
...,...,...
20108,ffd22b2ff47d4994b22f1fd7357507b0,"{'id': '23cec260c9b14d3bacae76c27d8db75e', 'ur..."
20109,fff7b77a40a247fb853b651b01c8f0db,"{'id': 'f1e5bd76a1074c09a5ec5c32cb880ccc', 'ur..."
20110,fff7b77a40a247fb853b651b01c8f0db,"{'id': '3fe7933f349848f08644cd85fea2938c', 'ur..."
20111,fff7b77a40a247fb853b651b01c8f0db,"{'id': '0e8728a2cda04e43a43f6f7b8258cbe6', 'ur..."


In [35]:
# 现在终于可以将所有信息重新按id分组，并将final_info列中的字典合并成一个列表，形成最终的images列
df_pics_grouped = (
    df_pics_to_group
    .groupby('id', as_index=False)
    .agg({'final_info': list})
)
df_pics_grouped.rename(columns={'final_info': 'images'}, inplace=True)
df_pics_grouped

,id,images
0,003895c08aa84c2a98d3d1696db7ffcf,"[{'id': 'f415bfc279cc411a96d354ec220aecf0', 'u..."
1,0043E78E1BCD4F11902DCA3C76607652,[{'id': 'e23f521d-556e-4ba4-8566-af692a6dcb37'...
2,004658C82423474EB46AA00F6BEEDAFC,[{'id': '44144ea9-053d-4c93-947a-950b542a43a4'...
3,0053DF2D67D346059E75B0729A69FE5B,[{'id': '7c1df671-ed26-4c3e-bf92-682fe10c43f6'...
4,005C79C1C40F4AB9AA9183E5B383E158,"[{'id': '1025b8b89e134b449abfac567fc59787', 'u..."
...,...,...
4087,fed05487276941e4bd2f329f373b4813,[{'id': '8a5890db-8f6a-4f97-b717-e61d4151563c'...
4088,ff413b38cf604dc6bd8667dced563377,"[{'id': '30529ae57a6c47229ee99a9ab5a2a539', 'u..."
4089,ffc96d5649c149ccb391212191c620f1,"[{'id': 'a98efdba0fbd4fed96bb853f8dc045e4', 'u..."
4090,ffd22b2ff47d4994b22f1fd7357507b0,"[{'id': '44fca8152939491f88d2040296f2d601', 'u..."


In [36]:
df_pics_online_urls = df_pics_final[['id', 'online_url']].copy()
df_pics_online_urls.rename(columns={'online_url': 'onlineImageUrls'}, inplace=True)
df_pics_online_urls_grouped = (
    df_pics_online_urls
    .groupby('id', as_index=False)
    .agg({'onlineImageUrls': list})
)
df_pics_online_urls_grouped

,id,onlineImageUrls
0,003895c08aa84c2a98d3d1696db7ffcf,[http://www.wwsdw.net/sdimg/back/picture/d40b5...
1,0043E78E1BCD4F11902DCA3C76607652,[http://www.wwsdw.net/sdimg/picture/shandong/3...
2,004658C82423474EB46AA00F6BEEDAFC,[http://www.wwsdw.net/sdimg/picture/shandong/3...
3,0053DF2D67D346059E75B0729A69FE5B,[http://www.wwsdw.net/sdimg/picture/shandong/3...
4,005C79C1C40F4AB9AA9183E5B383E158,[http://www.wwsdw.net/sdimg/picture/shandongne...
...,...,...
4087,fed05487276941e4bd2f329f373b4813,[http://www.wwsdw.net/sdimg/picture/shandong/3...
4088,ff413b38cf604dc6bd8667dced563377,[http://www.wwsdw.net/sdimg/back/picture/e6b6e...
4089,ffc96d5649c149ccb391212191c620f1,[http://www.wwsdw.net/sdimg/picture/shandongne...
4090,ffd22b2ff47d4994b22f1fd7357507b0,[http://www.wwsdw.net/sdimg/picture/shandongne...


In [37]:
df_pics_to_merge = df_pics_grouped.merge(df_pics_online_urls_grouped, on='id', how='left')
df_merged.drop(columns=['images'], inplace=True)
df_final = df_merged.merge(df_pics_to_merge, on='id', how='left')


In [38]:
df_final = df_final[['id', 'name', 'era', 'culture', 'time', 'dimensionsDesc',
       'structuredDimensions', 'fullDesc', 'features', 'excavationLocation',
       'currentLocation', 'excavationDate', 'sourceCitation', 'collectionInfo',
       'images', 'shape_type', 'onlineImageUrls', 
       
       'categoryName', 'clickCounts', 'collectedCounts',
       'collectionLevel', 'collectionTexture', 'collectionUnit',
       'collectionsCategory', 'fAudio', 'fVideo', 'imgUrl', 'isHighQuality',
       'museumName', 'threeUrl','pics', 'raw_excavation_info', 'raw_size_in_desc',
       'other_info',  ]].copy()
df_final = df_final.replace({pd.NA: None, np.nan: None})

In [39]:
final_data = df_final.to_dict(orient='records')


In [40]:
DATA_OUTPUT_PATH = DATA_DIR / "wwsd_full_data.json"
with open(DATA_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_data, f, indent=4, ensure_ascii=False)

In [41]:
df_final_simple = df_final[['id', 'name', 'era', 'culture', 'time', 'dimensionsDesc',
       'structuredDimensions', 'fullDesc', 'features', 'excavationLocation',
       'currentLocation', 'excavationDate', 'sourceCitation', 'collectionInfo',
       'images', 'shape_type', 'onlineImageUrls', ]].copy()
final_data_simple = df_final_simple.to_dict(orient='records')
DATA_OUTPUT_PATH = DATA_DIR / "wwsd_data.json"
with open(DATA_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_data_simple, f, indent=4, ensure_ascii=False)